# CSI bidding strategy by state  *(Python)*
Standalone twin of the dashboard's `state_summary()` logic (page 9). How many bids per state, win/loss, margins, competitive intensity, and the job-type / bid-quality mix by state. Win = CSI's own bid ranked 1st (row-derived, 117/603).

In [ ]:
from pathlib import Path
import warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 50)

# --- absolute paths (edit BASE if you move the project) ----------------------
BASE = Path("/Users/t/Library/Mobile Documents/com~apple~CloudDocs/00 MSBA + Coding/"
            "00_summer 2026/00_CSI_ELP/01_Summer_2026")
OUT  = BASE / "02_modules/sweet_spot/outputs"
base = pd.read_csv(OUT / "2026-06-04_project_base.csv")
arch = pd.read_csv(OUT / "2026-06-04_project_archetypes.csv")[["Contract_Id", "rule_archetype"]]
df = base.merge(arch, on="Contract_Id", how="left")
print("merged:", df.shape, "| archetypes:", df["rule_archetype"].notna().sum())

In [ ]:
b = df[df["CSI_Participated"].astype(str).str.upper()=="TRUE"].copy()
b["is_win"] = b["CSI_Won"].astype(str).str.upper()=="TRUE"
b["is_win_well"] = b["CSI_Bid_Quality"].isin({"Excellent Win","Acceptable Win"})
b["is_bomb"] = b["CSI_Bid_Quality"]=="Bomb"
b["margin"]=pd.to_numeric(b["CSI_Margin_Pct"],errors="coerce")
b["nb"]=pd.to_numeric(b["Num_Valid_Bidders"],errors="coerce")
print("CSI bids:", len(b), "| wins:", int(b["is_win"].sum()), "(expect 603 / 117)")

### State scorecard

In [ ]:
ss = b.groupby("State").apply(lambda g: pd.Series({
    "n_bid":len(g), "n_won":int(g["is_win"].sum()), "n_lost":len(g)-int(g["is_win"].sum()),
    "win_%":round(g["is_win"].mean()*100,1), "win_well_%":round(g["is_win_well"].mean()*100,1),
    "bomb_%":round(g["is_bomb"].mean()*100,1), "portfolio":g.loc[g["is_win"],"Project_Size_Dollars"].sum(),
    "avg_size":round(g["Project_Size_Dollars"].mean()), "med_margin":round(g["margin"].median(),2),
    "avg_bidders":round(g["nb"].mean(),1)}), include_groups=False).sort_values("n_bid", ascending=False)
print(ss.to_string())
print("\nsum n_bid:", int(ss["n_bid"].sum()))

### Bids vs wins, win rate, portfolio, competitive intensity

In [ ]:
STATE = {"Minnesota":"#1f4e79","North Dakota":"#2c7fb8","South Dakota":"#66a61e","Iowa":"#e67e22","Nebraska":"#7b3294","Montana":"#999999"}
cols = [STATE.get(s,"#888") for s in ss.index]
fig,ax=plt.subplots(2,2,figsize=(13,8))
ax[0,0].bar(ss.index, ss["n_bid"], color="#bdbdbd", label="bids"); ax[0,0].bar(ss.index, ss["n_won"], color="#1f4e79", label="wins")
ax[0,0].set_title("Bids vs wins"); ax[0,0].legend(); ax[0,0].tick_params(axis="x",rotation=30)
ax[0,1].bar(ss.index, ss["win_%"], color=cols); ax[0,1].axhline(19.4,color="gray",ls=":"); ax[0,1].set_title("Win % by state"); ax[0,1].tick_params(axis="x",rotation=30)
ax[1,0].bar(ss.index, ss["portfolio"]/1e6, color=cols); ax[1,0].set_title("Portfolio won ($M)"); ax[1,0].tick_params(axis="x",rotation=30)
ax[1,1].bar(ss.index, ss["avg_bidders"], color=cols); ax[1,1].set_title("Avg valid bidders (intensity)"); ax[1,1].tick_params(axis="x",rotation=30)
fig.tight_layout(); plt.show()

### Archetype mix & bid-quality mix by state (% of bids)

In [ ]:
order = ss.index.tolist()
am = b.groupby(["State","rule_archetype"]).size().unstack(fill_value=0).reindex(order)
am_pct = am.div(am.sum(axis=1),axis=0)*100
qm = b.assign(q=b["CSI_Bid_Quality"].fillna("Unknown")).groupby(["State","q"]).size().unstack(fill_value=0).reindex(order)
qm_pct = qm.div(qm.sum(axis=1),axis=0)*100
fig,ax=plt.subplots(1,2,figsize=(14,5))
am_pct.plot.bar(stacked=True, ax=ax[0], colormap="tab20"); ax[0].set_title("Archetype mix by state (%)"); ax[0].legend(fontsize=7, bbox_to_anchor=(1,1))
qm_pct.plot.bar(stacked=True, ax=ax[1], colormap="RdYlGn_r"); ax[1].set_title("Bid-quality mix by state (%)"); ax[1].legend(fontsize=7, bbox_to_anchor=(1,1))
fig.tight_layout(); plt.show()

## Takeaway
MN (268 bids/19%) and ND (218/22%) dominate the book; SD bids are far larger on average. The interactive version is **dashboard page 9**. Mirrors `utils/data.py state_summary / archetype_by_state / quality_by_state`.